# Session 2, Part 1: uncertainty and sensitivity analysis

Run each cell with **Shift+Enter**, from the top. Cells marked **Your turn** are exercises at level **1** (change a value), **2** (a few lines of code), or **3** (advanced, optional). Every exercise cell already runs as is. Each exercise has a hint, and a solution that stays hidden until you click **Show solution**.

Inputs to a TEA or LCA are never known exactly: prices change, characterization factors depend on the data source, equipment costs are estimates. Two questions follow.

- **Uncertainty analysis:** given plausible ranges for the inputs, what is the *range* of the results?
- **Sensitivity analysis:** which inputs drive the result, so that you know where better data matter?

QSDsan does both on top of your system, so the same model that gives you a single TEA and LCA answer also gives you a distribution of answers. We use the aerobic and anaerobic plants from session 1 as the example.

In [ ]:
# Google Colab only: installs QSDsan the first time. On your own computer this cell does nothing.
# After the install, restart the session (Runtime, Restart session) and run the notebook again from the top.
import sys
if 'google.colab' in sys.modules:
    try:
        import qsdsan
    except ImportError:
        %pip install -q qsdsan==1.5.3 exposan==1.5.3
        print('Installed. Now restart the session (Runtime, Restart session) and run the notebook again from the top.')
    else:
        import biosteam as bst
        bst.preferences.graphviz_format = 'png'   # Colab does not draw the default SVG diagrams

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import qsdsan as qs
from chaospy import distributions as shape   # probability distributions for the inputs
print(f'QSDsan {qs.__version__}')

qs.main_flowsheet.set_flowsheet('s2p1')

## 1. The question: aerobic or anaerobic?

In session 1 we compared an aerobic and an anaerobic plant with fixed inputs: the aerobic plant was cheaper, the anaerobic plant had the lower carbon footprint. But the inputs are not fixed. Wastewater strength varies, prices change, and the capital costs are estimates. Two questions follow:

1. **How likely** is the anaerobic plant to be cheaper, or to have the lower footprint, once the inputs vary? (uncertainty analysis)
2. **Which inputs** drive the difference between the plants? (sensitivity analysis)

This cell repeats the setup from session 1 part 2: both plants, their TEA, and their LCA.

In [ ]:
from qsdsan.utils import create_example_treatment_systems

qs.ImpactIndicator('GlobalWarming', alias='GWP', unit='kg CO2-eq')
aer_sys, ana_sys = create_example_treatment_systems()
aer, ana = aer_sys.units[0], ana_sys.units[0]
both = qs.System('both', path=(aer_sys, ana_sys))    # one system holding both plants, so one model can vary both
both.simulate()

ww_aer, ww_ana = aer.ins[0], ana.ins[0]
ww_aer.price = ww_ana.price = -0.0006                # a treatment fee received, USD/kg (about 0.60 USD/m3)
ana.ins[1].price = 0.90                              # sodium bicarbonate, USD/kg
qs.PowerUtility.price = 0.08                         # USD/kWh
tea_aer = qs.TEA(aer_sys, discount_rate=0.05, lifetime=20)
tea_ana = qs.TEA(ana_sys, discount_rate=0.05, lifetime=20)

# Life cycle inventory (illustrative values, as in session 1)
Concrete = qs.ImpactItem('Concrete', functional_unit='m3', GWP=300.)
Steel = qs.ImpactItem('Steel', functional_unit='kg', GWP=2.)
Trucking = qs.ImpactItem('Trucking', functional_unit='kg*km', GWP=1e-4)
electricity = qs.ImpactItem('electricity', functional_unit='kWh', GWP=0.5)
for u, concrete_m3, steel_kg in ((aer, 300, 30000), (ana, 400, 40000)):
    u.construction = [
        qs.Construction(linked_unit=u, item=Concrete, quantity=concrete_m3, quantity_unit='m3', lifetime=30),
        qs.Construction(linked_unit=u, item=Steel, quantity=steel_kg, quantity_unit='kg', lifetime=30)]
    u.transportation = [qs.Transportation(
        linked_unit=u, item=Trucking, load_type='mass', load=u.outs[1].F_mass * 24, load_unit='kg',
        distance=50, distance_unit='km', interval=1, interval_unit='day')]
qs.StreamImpactItem(linked_stream=aer.outs[1], GWP=0.2)             # sludge handling
qs.StreamImpactItem(linked_stream=ana.outs[1], GWP=0.2)
biogas_item = qs.StreamImpactItem(linked_stream=ana.outs[2], GWP=-2.0)   # biogas displaces natural gas

lifetime = 20
lca_aer = qs.LCA(system=aer_sys, lifetime=lifetime, simulate_system=False,
                 electricity=lambda: aer.power_utility.rate * 24 * 365 * lifetime)   # a function, so it follows the simulation
lca_ana = qs.LCA(system=ana_sys, lifetime=lifetime, simulate_system=False,
                 electricity=lambda: ana.power_utility.rate * 24 * 365 * lifetime)
print(f'Baseline equivalent annual cost: aerobic {tea_aer.EAC:,.0f} USD/yr, anaerobic {tea_ana.EAC:,.0f} USD/yr (negative = net gain)')
print(f'Baseline GWP: aerobic {lca_aer.get_normalized_impacts(ww_aer, normalize_by="volume")["GlobalWarming"]:.3f}, '
      f'anaerobic {lca_ana.get_normalized_impacts(ww_ana, normalize_by="volume")["GlobalWarming"]:.3f} kg CO2-eq per m3 treated')

## 2. Model, parameters, and metrics

A **`Model`** wraps a system. You add:

- **parameters**: the uncertain inputs. Each has a baseline, a probability distribution, a setter function that applies a value to the system, and a `kind`:
  - `'coupled'`: the value changes the simulation itself (influent strength, a process efficiency), so the system is re-simulated for every sample;
  - `'isolated'`: it only affects cost or impact calculations (a price, a characterization factor), so no re-simulation is needed;
- **metrics**: the outputs you track, each a function that returns a number.

Our model wraps the system with both plants, so each sample gives the same inputs to both and we can compare them directly.

In [ ]:
model = qs.Model(both)
param = model.parameter
metric = model.metric
Substrate = aer.components.Substrate

# Wastewater and process inputs: these change the simulation, so they are 'coupled'
@param(name='Influent COD', element='Wastewater', kind='coupled', units='mg/L',
       baseline=430, distribution=shape.Uniform(lower=300, upper=3000))
def set_cod(cod):
    for w in (ww_aer, ww_ana):
        w.imass['Substrate'] = (cod / 1000 * 4000 / 24) / Substrate.i_COD    # 4,000 m3/d

@param(name='X_oxid', element=aer, kind='coupled', units='-',
       baseline=aer.X_oxid, distribution=shape.Uniform(lower=0.85, upper=0.99))
def set_x_oxid(x):
    aer.X_oxid = x          # fraction of substrate oxidized, which sets the aeration electricity

@param(name='Heat recovery efficiency', element=ana, kind='coupled', units='-',
       baseline=0.8, distribution=shape.Uniform(lower=0.6, upper=0.9))
def set_hx(x):
    ana.HX_eff = x          # sets the heating demand of the anaerobic plant

@param(name='Biogas energy price', element=ana, kind='coupled', units='USD/GJ',
       baseline=5, distribution=shape.Uniform(lower=3, upper=8))
def set_biogas_price(x):
    ana.energy_price = x / 1e6      # value of the biogas, USD/kJ

In [ ]:
# Cost inputs: they only enter the cost calculation, so they are 'isolated'
@param(name='Treatment fee', element='TEA', kind='isolated', units='USD/m3',
       baseline=0.60, distribution=shape.Uniform(lower=0.30, upper=1.00))
def set_fee(fee_per_m3):
    ww_aer.price = ww_ana.price = -fee_per_m3 / 1000      # USD/kg, negative = revenue

base = qs.PowerUtility.price
@param(name='Electricity price', element='TEA', kind='isolated', units='USD/kWh',
       baseline=base, distribution=shape.Triangle(lower=base*0.75, midpoint=base, upper=base*1.25))
def set_e_price(p):
    qs.PowerUtility.price = p

@param(name='Aerobic capital', element=aer, kind='isolated', units='USD',
       baseline=5e6, distribution=shape.Triangle(lower=4e6, midpoint=5e6, upper=6e6))
def set_aer_capital(c):
    aer.plant_capital = c

@param(name='Anaerobic capital', element=ana, kind='isolated', units='USD',
       baseline=5e6, distribution=shape.Triangle(lower=4e6, midpoint=5e6, upper=6e6))
def set_ana_capital(c):
    ana.plant_capital = c

# Life cycle inputs: characterization factors, also 'isolated'
base = Steel.CFs['GlobalWarming']
@param(name='Steel GWP', element='LCA', kind='isolated', units='kg CO2-eq/kg',
       baseline=base, distribution=shape.Uniform(lower=base*0.9, upper=base*1.5))
def set_steel_gwp(cf):
    Steel.CFs['GlobalWarming'] = cf

base = electricity.CFs['GlobalWarming']
@param(name='Electricity GWP', element='LCA', kind='isolated', units='kg CO2-eq/kWh',
       baseline=base, distribution=shape.Uniform(lower=base*0.7, upper=base*1.3))
def set_e_gwp(cf):
    electricity.CFs['GlobalWarming'] = cf

@param(name='Biogas credit GWP', element='LCA', kind='isolated', units='kg CO2-eq/kg',
       baseline=-2.0, distribution=shape.Uniform(lower=-3.0, upper=-1.0))
def set_biogas_credit(cf):
    biogas_item.CFs['GlobalWarming'] = cf       # impact avoided per kg of biogas

In [ ]:
def gwp_per_m3(lca, unit):
    return lca.get_normalized_impacts(unit.ins[0], normalize_by='volume')['GlobalWarming']

@metric(name='EAC aerobic', units='USD/yr', element='TEA')
def get_eac_aer():
    return tea_aer.EAC

@metric(name='EAC anaerobic', units='USD/yr', element='TEA')
def get_eac_ana():
    return tea_ana.EAC

@metric(name='Cost advantage of anaerobic', units='USD/yr', element='TEA')
def get_cost_adv():
    return tea_aer.EAC - tea_ana.EAC          # positive: the anaerobic plant is cheaper

@metric(name='GWP aerobic', units='kg CO2-eq/m3', element='LCA')
def get_gwp_aer():
    return gwp_per_m3(lca_aer, aer)

@metric(name='GWP anaerobic', units='kg CO2-eq/m3', element='LCA')
def get_gwp_ana():
    return gwp_per_m3(lca_ana, ana)

@metric(name='Carbon advantage of anaerobic', units='kg CO2-eq/m3', element='LCA')
def get_carbon_adv():
    return gwp_per_m3(lca_aer, aer) - gwp_per_m3(lca_ana, ana)     # positive: the anaerobic plant has the lower footprint

model.show()

## 3. Uncertainty analysis

The steps are always the same: **sample** the inputs from their distributions, **load** the samples, **evaluate** the model (one simulation plus TEA and LCA per sample), and look at the results table. We use Latin hypercube sampling (`rule='L'`), which covers the input space more evenly than plain random sampling.

In [ ]:
N = 1000
np.random.seed(328)                      # a fixed seed makes the run reproducible
samples = model.sample(N=N, rule='L')
model.load_samples(samples)

t0 = time.time()
model.evaluate()
print(f'{N} evaluations took {time.time() - t0:.1f} s')

n_par = len(model.parameters)
results = model.table.iloc[:, n_par:].copy()
results.columns = results.columns.get_level_values(1)        # keep the names with units
results.quantile([0.05, 0.5, 0.95]).round(2).T               # median and a 90% interval for each metric

In [ ]:
# The decision questions: in what share of the samples does the anaerobic plant win?
cost_adv, carbon_adv = results.iloc[:, 2], results.iloc[:, 5]
print(f'Anaerobic cheaper than aerobic:        {(cost_adv > 0).mean():.0%} of samples')
print(f'Anaerobic lower carbon than aerobic:   {(carbon_adv > 0).mean():.0%} of samples')

In [ ]:
# The joint distribution of the two advantages. Right of 0: anaerobic is cheaper. Above 0: anaerobic has the lower footprint.
fig, ax = qs.stats.plot_uncertainties(model, x_axis=model.metrics[2], y_axis=model.metrics[5],
                                      kind='kde-kde', center_kws={'fill': True})
fig

Over these ranges, the carbon ranking never changes (the anaerobic plant always has the lower footprint) and neither does the cost ranking (the aerobic plant is always cheaper). That is useful to know: **the trade-off is robust to these uncertainties**. Sensitivity analysis will show which inputs move the size of the gap.

**How many samples?** Too few samples give unstable percentiles. A simple check is to repeat the run with more samples and see when the numbers stop changing.

**Your turn 1 (level 1).** Repeat the uncertainty analysis with N = 50, 500, and 5000 samples (same seed). For each, print the mean and the 5th and 95th percentiles of the **cost advantage of anaerobic**. At what N do the numbers stop changing noticeably?

<details><summary>Hint</summary>

Write a loop over the three sample sizes, like the one for N = 50. The mean settles early; the 5th and 95th percentiles need more samples.

</details>

In [ ]:
def run_ua(N, seed=328):
    np.random.seed(seed)
    model.load_samples(model.sample(N=N, rule='L'))
    model.evaluate()
    return model.table.iloc[:, n_par + 2]      # the cost advantage column

for N in (50,):                                # Your turn: add 500 and 5000
    adv = run_ua(N)
    print(f'N = {N:>5}: mean {adv.mean():>10,.0f}, 5th {adv.quantile(0.05):>10,.0f}, 95th {adv.quantile(0.95):>10,.0f}')
run_ua(1000);   # leave the model with the N = 1000 results for the cells below

<details><summary>Show solution</summary>

```python
def run_ua(N, seed=328):
    np.random.seed(seed)
    model.load_samples(model.sample(N=N, rule='L'))
    model.evaluate()
    return model.table.iloc[:, n_par + 2]      # the cost advantage column

for N in (50, 500, 5000):
    adv = run_ua(N)
    print(f'N = {N:>5}: mean {adv.mean():>10,.0f}, 5th {adv.quantile(0.05):>10,.0f}, 95th {adv.quantile(0.95):>10,.0f}')
run_ua(1000);   # leave the model with the N = 1000 results for the cells below
```

</details>

**Your turn 2 (level 2).** The anaerobic plant is expected to win on cost for very strong wastewater. Widen the distribution of the influent COD to between 300 and 15,000 mg/L (it is the first parameter, `model.parameters[0]`), re-run with N = 1000, and print the share of samples in which the anaerobic plant is cheaper. Then restore the original range (300 to 3000).

<details><summary>Hint</summary>

Assign a new chaospy distribution to the parameter's `distribution` attribute, then sample and evaluate again.

</details>

In [ ]:
cod = model.parameters[0]
# Your turn: set a new distribution, e.g. cod.distribution = shape.Uniform(lower=..., upper=...)
np.random.seed(328)
model.load_samples(model.sample(N=1000, rule='L'))
model.evaluate()
adv = model.table.iloc[:, n_par + 2]
print(f'Anaerobic cheaper in {(adv > 0).mean():.0%} of samples')
cod.distribution = shape.Uniform(lower=300, upper=3000)      # restore the original range
run_ua(1000);

<details><summary>Show solution</summary>

```python
cod = model.parameters[0]
cod.distribution = shape.Uniform(lower=300, upper=15000)
np.random.seed(328)
model.load_samples(model.sample(N=1000, rule='L'))
model.evaluate()
adv = model.table.iloc[:, n_par + 2]
print(f'Anaerobic cheaper in {(adv > 0).mean():.0%} of samples')
cod.distribution = shape.Uniform(lower=300, upper=3000)      # restore the original range
run_ua(1000);
```

</details>

## 4. Sensitivity analysis

Several methods answer "which inputs matter?". They differ in cost and in what they tell you.

| Method | Function | Extra simulations | What it gives |
|---|---|---|---|
| Rank correlation (Spearman) | `get_correlations` | none (reuses the table above) | direction and strength of monotonic relationships |
| Morris elementary effects | `morris_analysis` | N x (k + 1) | screening: which inputs matter, and which act nonlinearly or by interaction |
| Sobol indices | `sobol_analysis` | N x (2k + 2) | share of the output variance from each input (first order) and including interactions (total) |
| FAST | `fast_analysis` | N x k | first-order shares, cheaper than Sobol |

Here k is the number of parameters. We use the first two, which are the cheapest. Sobol and FAST work the same way (generate their own samples, evaluate, analyze) and are described in tutorial 9; they pay off for the few inputs that survive screening.

### 4.1. Rank correlation

In [ ]:
r_df, p_df = qs.stats.get_correlations(model, kind='Spearman')
r_df.iloc[:, [2, 5]].round(2)        # the columns for the two advantages

In [ ]:
fig, ax = qs.stats.plot_correlations(r_df, metrics=(model.metrics[2], model.metrics[5]))   # bubble size and color show the strength
fig

The cost advantage is mostly driven by the heat recovery efficiency and the influent strength, with the two capital costs next. The carbon advantage follows the influent strength (more COD gives more biogas) and the biogas credit. The treatment fee does not matter for the *difference* because both plants receive it.

A correlation of 0 does not mean no effect for a nonlinear model, so we check with a method that does not assume a monotonic relationship.

### 4.2. Morris screening

Morris changes one input at a time along random paths and measures the average absolute effect (mu*) and how much that effect varies (sigma). A large sigma compared with mu* suggests nonlinearity or interactions. It uses its own sample design, so we generate new samples.

In [ ]:
inputs = qs.stats.define_inputs(model)
samples = qs.stats.generate_samples(inputs, kind='Morris', N=10, seed=328)
print(f'{len(samples)} samples')
model.load_samples(samples)
model.evaluate()
morris = qs.stats.morris_analysis(model, inputs, seed=328, nan_policy='fill_mean')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, m in zip(axes, (model.metrics[2], model.metrics[5])):
    qs.stats.plot_morris_results(morris, metric=m, ax=ax, label_kind='name')
    ax.set_title(m.name)
fig.tight_layout()
fig

### 4.3. Do the methods agree?

The table scores the inputs for one metric under the two methods. Each column is divided by its largest value, so that 1 means the most influential input and 0 means no influence. Agreement on the top inputs is a good sign.

In [ ]:
def compare(i):
    '''Score the inputs for metric number i under two methods, as a share of the largest value.'''
    name = model.metrics[i].name
    scores = pd.DataFrame({
        'Spearman |r|': r_df.iloc[:, i].abs().values,
        'Morris mu*': morris[name]['mu_star'].values,
    }, index=[p.name for p in model.parameters])
    return (scores / scores.max()).round(2).sort_values('Morris mu*', ascending=False)

compare(2)    # cost advantage of anaerobic

**Your turn 3 (level 1).** Run `compare` for the **carbon advantage** (metric number 5). Which input matters most? Do the two methods agree on the order of the top three inputs? Which inputs from the cost side drop out?

<details><summary>Hint</summary>

Metric 2 is the cost advantage and metric 5 is the carbon advantage. Inputs with almost no effect have scores close to zero, so their order is mostly noise.

</details>

In [ ]:
compare(2)    # Your turn: change the metric number

<details><summary>Show solution</summary>

```python
compare(5)
```

</details>

## 5. When the model is expensive

The plant above evaluates in milliseconds. Real models do not. EXPOsan's `bwaise` model for sanitation system A (a published study) has **137 uncertain parameters and 26 metrics**, and every sample is a full simulation with TEA and LCA.

In [ ]:
from exposan import bwaise as bw
import exposan.bwaise.models as bwm

bw.load()
bw_model = bwm.create_model('A')
print(f'{len(bw_model.parameters)} parameters, {len(bw_model.metrics)} metrics')

Time a small run first, so that you know the cost before you commit to a large one.

In [ ]:
N_SMALL = 20
np.random.seed(328)
t0 = time.time()
bw_model.load_samples(bw_model.sample(N=N_SMALL, rule='L'))
bw_model.evaluate()
per_sample = (time.time() - t0) / N_SMALL
print(f'{N_SMALL} samples took {per_sample * N_SMALL:.0f} s: {per_sample:.2f} s per sample, '
      f'so 1,000 samples would take about {per_sample * 1000 / 60:.0f} minutes')

In [ ]:
# The number of simulations each method needs (k = number of parameters)
k = len(bw_model.parameters)
runs = {'Monte Carlo, N = 1000': 1000,
        'Morris, N = 10': 10 * (k + 1),
        'FAST, N = 100': 100 * k,
        'Sobol, N = 64': 64 * (2 * k + 2)}
for name, n in runs.items():
    print(f'{name:<24} {n:>7,} simulations, about {n * per_sample / 3600:5.1f} hours')

Sobol indices for all 137 inputs would take many hours, so you screen first (rank correlation, then Morris) and apply the expensive methods only to the inputs that matter. A large analysis of a model like this is better run outside a live session, for example overnight. Save the results table to a file, so that you run it once and analyze it many times, and load it back into the model when you need it:

In [ ]:
import os, tempfile

path = os.path.join(tempfile.mkdtemp(), 'bwaise_small_run.csv')
bw_model.table.to_csv(path)                                          # save the results table
bw_model.table = pd.read_csv(path, header=[0, 1], index_col=0)       # load it back into the model
print(f'{len(bw_model.table)} samples loaded from the saved file')

## Choosing and planning an analysis

1. **Decide the inputs and their distributions first.** The result is only as good as the ranges. Use data, literature, or expert judgment, and keep each range inside the region where the model is valid.
2. **Time a small run** before you commit: the cost is (number of simulations) x (time per simulation).
3. **Screen, then focus.** Rank correlation costs nothing extra, Morris screens many inputs cheaply, and Sobol and FAST are for the few inputs that matter (see tutorial 9). For a model that is expensive or has many inputs, a staged workflow is common:
   1. Run **Morris** on all inputs with wide but plausible ranges.
   2. **Fix the clearly negligible inputs** at their baseline values.
   3. Run the **Monte Carlo** analysis (and Spearman) on the remaining inputs with many more samples.
   4. Run **Sobol or FAST** on the top few, if you need variance shares or interactions.

   Conditions: keep the ranges plausible, because Morris uses the bounds, not the shape of the distributions, and absurd extremes can break the simulation. Call an input negligible only if mu* is small for **every** metric you care about, and keep inputs with a large sigma, which signals an interaction. Fixing inputs understates the output uncertainty, so say so when you report it. Use enough Morris trajectories for the ranking to settle (`morris_till_convergence` checks this), and remember that Morris, Sobol, and FAST assume independent inputs. When the model is cheap, as the two plants above, running Monte Carlo and Morris on all inputs is simpler.
4. **Use `'isolated'` for inputs that only affect cost or impact calculations**, and `'coupled'` only when the simulation must be redone.
5. **Check convergence** by increasing N, and fix the seed so that others can reproduce your numbers.
6. **Save the results table** (`model.table.to_csv(...)`), so that heavy runs are done once and analyzed many times.

## Where to go next

- Tutorial: [Uncertainty and Sensitivity Analyses](https://qsdsan.readthedocs.io/en/latest/tutorials/9_Uncertainty_and_Sensitivity_Analyses.html), and the [stats module API](https://qsdsan.readthedocs.io/en/latest/api/stats.html) for all analysis and plotting functions.
- Published examples: the `models.py` files of EXPOsan modules (for example `bwaise`, `reclaimer`).
- Next in this session: dynamic simulation.